# **Label-Agnostic Bayesian Optimization for Intrusion Detection (Label Trainer — v3.5.4)**

**Final Project — PCMLAI, Imperial College London**

This project develops a resilient Intrusion Detection System (IDS) designed to detect **unknown, evolving, and imbalanced cyber threats** in network traffic. Using the **UNSW-NB15 dataset**, the system learns to distinguish **benign traffic** from a wide range of attack categories under two challenging but realistic conditions: **class imbalance** and **label noise**.

To deliver high recall without overwhelming analysts with false alerts, we apply **Bayesian Optimization (BO)** to fine-tune a **LightGBM** classifier. This approach achieves \~0.95 **PR-AUC**, demonstrating reliable detection of rare but high-risk attacks. As a benchmark, a compact **1D Convolutional Neural Network (CNN)** was tested, but tree-based methods provided superior performance and stability for structured network data.

The workflow includes **interpretability (SHAP)** and **ensemble methods** (bagging, soft voting, stacking) to enhance both accuracy and trust. All results are versioned in a **staging pipeline** with manifest tracking, ensuring that experiments remain reproducible, auditable, and ready for operational deployment.

---

### **Problem Statement**

* **Task:** Train a robust IDS on tabular traffic data subject to **label uncertainty** and **severe imbalance**.
* **Objective:** Optimise **PR-AUC** through stratified cross-validation; tune the decision threshold $\tau$ for **F1-maximisation** or operational constraints (e.g., high precision regimes).

---

### **Methods & Pipeline**

* **Backbone Classifier:** LightGBM (with `class_weight="balanced"`).
* **Hyperparameter Optimisation:**

  * Manual grid search (resumable).
  * Randomised search.
  * **Bayesian Optimisation** (`BayesSearchCV`).
  * **Enhanced BO**: ask–tell API, warm start, batched EI, patience criteria, resumable runs.
* **Benchmark Model:** Lightweight 1D-CNN treating tabular rows as sequences.
* **Ensemble Strategies:**

  * Bagging (Random Forests).
  * Soft voting (probability averaging).
  * Stacking (meta-learner: logistic regression).
* **Reproducibility:**

  * Stage manifests (`staging/<stage>/manifest.json`).
  * Auto-generated `README.md` and `model_card.md` documenting best models.

---

### **PCMLAI Module Links**

* **Module 3 – Probabilistic Modelling**
  PR-curves, threshold tuning under imbalance, F1 vs operational trade-offs.

* **Module 10 – Model Selection & Black-box Optimisation**
  Grid search, Bayesian Optimisation, ensembles, early stopping.

* **RL/Bandits Connection**
  BO’s acquisition function parallels bandit trade-offs: **exploration** vs **exploitation**.

---

### ✅ **Key Takeaways**

* **Bayesian Optimisation** efficiently converges to strong hyperparameters with fewer trials.
* **Enhanced BO** (resume + patience) improves robustness under long runs.
* **CNN** provides a neural baseline but underperforms tree-based methods for tabular IDS.
* **Ensemble methods** stabilise performance across folds and noisy labels.
* **SHAP analysis** confirms feature stability and interpretable model behaviour.

> The final system offers a **label-agnostic, interpretable, and automated IDS pipeline** — combining high detection performance with operational reproducibility for deployment in real-world SOC environments.



In [ ]:
# ======================================================
# Section 2 — Data Loading (must run before schema inference)
# Rationale:
#   • Ensure a canonical dataframe `data` exists prior to deriving schema and building the preprocessor.
#   • Provide X/y and stratified train/test split for downstream sections.
# ======================================================
print(">>> Section 2: Data Loading (pre-schema)")

import os
import pandas as pd
from sklearn.model_selection import train_test_split

# --- Configure dataset path and target column ---
DATA_PATH = os.environ.get("DATA_PATH", "archive/Payload_data_UNSW.csv")  # change if needed
TARGET_COL = os.environ.get("TARGET_COL", "label")               # change if needed

# Skip if already loaded in the session
if 'data' in globals() and isinstance(data, pd.DataFrame):
    print("[Info] Found existing `data` in memory; skipping reload.")
else:
    if not os.path.exists(DATA_PATH):
      raise FileNotFoundError(f"Dataset not found at {DATA_PATH}. Set DATA_PATH to the correct location.")
    data = pd.read_csv(DATA_PATH)
    print(f"[Load] data.shape={data.shape} from {DATA_PATH}")

if TARGET_COL not in data.columns:
    raise KeyError(f"Target column '{TARGET_COL}' not found. Available columns include: {list(data.columns)[:12]} ...")

# Split features/labels
y = data[TARGET_COL]
X = data.drop(columns=[TARGET_COL])

# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Sanity prints
print(f"[Split] X_train={X_train.shape}, X_test={X_test.shape}")
print("[Label distribution] y_train:")
print(y_train.value_counts(normalize=True).round(3))


>>> Section 2: Data Loading (pre-schema)
[Info] Found existing `data` in memory; skipping reload.
[Split] X_train=(63904, 1504), X_test=(15977, 1504)
[Label distribution] y_train:
label
normal            0.263
generic           0.220
exploits          0.175
fuzzers           0.159
reconnaissance    0.095
dos               0.043
backdoor          0.016
analysis          0.015
shellcode         0.014
worms             0.001
Name: proportion, dtype: float64


## 2. Reproducibility & Utilities

- Fixed seeds & thread caps → stable results.
- **Staging** per experiment → auditability & resume.
- Helpers:
  - `sweep_f1`: grid search on \( \tau \) for max-F1 (Module 3).
  - `tpr_tnr_at_tau`: operational rates at τ.


In [22]:
# ======================================================
# Section 2 — Dataset Schema → Categorical/Numeric Splits (for LightGBM & CT)
# Rationale:
#   • Define categorical and numeric feature sets from the loaded dataframe.
#   • Print value_counts for object dtypes (analyst visibility, class drift checks).
#   • Provide cat_cols for LightGBM native categorical handling (Section 4).
#   • Build a ColumnTransformer aligned with these splits for general pipelines.
# Outputs:
#   • cat_cols, num_cols (globals)
#   • preprocessor (ColumnTransformer) with pandas output if supported
# ======================================================
print(">>> Section 2: Derive cat_cols/num_cols and refresh preprocessor")

import numpy as np
import pandas as pd

# Prefer 'data' as the canonical dataframe, else fall back to X if available
_df = None
for cand in ['data', 'X', 'X_train']:
    if cand in globals():
        _df = globals()[cand]
        print(f"[Info] Using dataframe from '{cand}' for schema inference.")
        break

if _df is None:
    raise RuntimeError("No dataframe found for schema inference. Define 'data' (preferred) or 'X'/'X_train' earlier.")

# Identify target label if present to exclude from features
_target = None
for t in ['label', 'y', 'target', 'Label']:
    if t in _df.columns:
        _target = t
        break

# Derive categorical and numeric features (exclude target where applicable)
cat_cols = _df.select_dtypes(include=['object','category']).columns.tolist()
num_cols = _df.select_dtypes(include=['number']).columns.tolist()
if _target in cat_cols:
    cat_cols.remove(_target)
if _target in num_cols:
    num_cols.remove(_target)

print(f"[Schema] |cat_cols|={len(cat_cols)} |num_cols|={len(num_cols)}")

# Print value_counts for categorical/object features (analyst visibility)
for c in cat_cols:
    try:
        print(f"[value_counts] {c}")
        print(_df[c].value_counts())
    except Exception as e:
        print(f"[value_counts] {c} skipped: {e}")

# Simple numeric outlier columns using z-score>3 (visibility only; no mutation)
_outlier_cols = []
for col in num_cols:
    s = _df[col]
    std = s.std(ddof=0)
    if std == 0 or pd.isna(std):
        continue
    z = (s - s.mean()) / std
    if (np.abs(z) > 3).sum() > 0:
        _outlier_cols.append(col)
print("[Outliers] numeric columns with any |z|>3:", _outlier_cols)

# Build/refresh ColumnTransformer aligned with these splits
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Version-safe OneHotEncoder arg for sparse_output vs sparse
try:
    _ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    _ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", _ohe, cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

# Ensure pandas output if supported
try:
    preprocessor.set_output(transform="pandas")
    print("[Info] preprocessor.set_output(transform='pandas') applied.")
except Exception as e:
    print("[Info] preprocessor.set_output(transform='pandas') not applied:", e)

# For LightGBM native categorical support (Section 4), cast original categorical columns to 'category' dtype if using 'data'
try:
    if 'data' in globals():
        for c in cat_cols:
            if globals()['data'][c].dtype.name not in ('category',):
                globals()['data'][c] = globals()['data'][c].astype('category')
        print("[Info] Casted 'data' categorical columns to pandas 'category' dtype for LightGBM.")
except Exception as e:
    print("[Info] Could not cast 'data' categories:", e)


>>> Section 2: Derive cat_cols/num_cols and refresh preprocessor
[Info] Using dataframe from 'data' for schema inference.
[Schema] |cat_cols|=1 |num_cols|=1503
[value_counts] protocol
protocol
tcp            42428
udp            29188
others          5657
ospf             521
sctp             345
gre               95
swipe             81
sun-nd            81
sep               81
mobile            81
unas              60
pim               50
micp              46
rdp               46
pipe              46
secure-vmtp       46
rsvp              46
sccopmce          46
snp               46
sps               46
nvp               46
vmtp              46
ax.25             46
leaf-2            46
crtp              46
crudp             46
dgp               46
egp               46
emcon             46
etherip           46
fire              46
ggp               46
gmtp              46
hmp               46
ib                46
ipip              46
iplt              46
ipv6              46
arp      

In [23]:
# ======================================================
# Cell 3 — Imports, Reproducibility, and Utility helpers
# Why:
#   • Determinism & thread control → stable, repeatable runs (Module 10).
#   • Staging helpers → audit trail (manifest, results, model).
#   • Threshold/metrics helpers → interpretable evaluation (Module 3).
# ======================================================
print(">>> Section 2: Imports + Reproducibility + Utilities")

import os, re, json, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import average_precision_score, precision_recall_curve, f1_score, confusion_matrix
from lightgbm import LGBMClassifier

# BO tooling (skopt): BayesSearchCV (high-level) and Optimizer (ask–tell)
from skopt import BayesSearchCV, Optimizer
from skopt.space import Real, Integer
from skopt.callbacks import CheckpointSaver
from joblib import Parallel, delayed    # CPU parallel CV

warnings.filterwarnings('ignore')       # cleaner notebook visuals for viva

# ---- Reproducibility & parallelism -----------------------------------------
RANDOM_STATE = 42                        # fixed seed for all RNG consumers
np.random.seed(RANDOM_STATE)
N_THREADS = max(1, (os.cpu_count() or 4) - 1)  # keep 1 CPU free for system
# Cap BLAS threads to avoid oversubscription (keeps laptop fans sane)
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(N_THREADS))

# ---- Staging root (all artifacts live here) --------------------------------
stage_root = Path('staging')
stage_root.mkdir(exist_ok=True, parents=True)

def stage_dir(name: str) -> Path:
    """Create (if needed) and return 'staging/<name>' directory."""
    d = stage_root / name
    d.mkdir(parents=True, exist_ok=True)
    return d

def save_stage(name: str, manifest: dict, results: pd.DataFrame | None = None, model=None):
    """
    Persist experiment state for reproducibility & audit:
      - manifest.json: single source of truth (metrics/params/dataset/seed).
      - results.csv:   optional search history (e.g., CV table).
      - model.joblib:  optional fitted estimator (tabular models).
    """
    sd = stage_dir(name)
    (sd/'manifest.json').write_text(json.dumps(manifest, indent=2))
    if isinstance(results, pd.DataFrame):
        (sd/'results.csv').write_text(results.to_csv(index=False))
    if model is not None:
        try:
            import joblib
            joblib.dump(model, sd/'model.joblib')
        except Exception as e:
            print(f"[WARN] Model save failed for {name}: {e}")
    print(f"[SAVE] Stage '{name}' saved -> {sd}")

def load_stage(name: str) -> dict | None:
    """
    Load an existing manifest for e.g. warm-starting or reporting.
    Returns None if missing/corrupt: presentation-safe behavior.
    """
    p = stage_dir(name) / 'manifest.json'
    if p.exists():
        try:
            return json.loads(p.read_text())
        except Exception:
            return None
    return None

def sweep_f1(y_true, y_proba, grid=np.linspace(0.05, 0.95, 19)):
    """
    Pick threshold τ on a coarse grid that maximizes F1.
    Why coarse? Stable, robust to tiny score noise; teachers like the clarity.
    """
    f1s = [(t, f1_score(y_true, (y_proba >= t).astype(int), zero_division=0)) for t in grid]
    t, f = max(f1s, key=lambda z: z[1])
    return float(f), float(t)

def tpr_tnr_at_tau(y_true, p_hat, tau: float):
    """
    Compute operational rates at τ:
      - TPR (recall): attack catch rate.
      - TNR (specificity): benign tolerance / low false alarm rate.
    """
    yb = (p_hat >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yb).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    tnr = tn / (tn + fp) if (tn + fp) else 0.0
    return float(tpr), float(tnr)

print("[STATUS] Utilities ready. N_THREADS=", N_THREADS)


# --- Flexible Feature Weighting (minimal patch) ---
# You can assign weights to any subset of features; unspecified features default to 1.0.
FEATURE_WEIGHTS = {  # e.g., {"ttl": 0.1, "payload_byte_1": 0.5}
    # "ttl": 0.1,
}
def _apply_feature_weights(df_or_arr, feature_names=None, weights=FEATURE_WEIGHTS):
    """
    Minimal intrusive scaler: multiply specified columns by given weights.
    - If df_or_arr is a pandas DataFrame: use column names directly.
    - If it's a NumPy array: use feature_names to locate columns.
    """
    try:
        import pandas as _pd
        if hasattr(df_or_arr, "columns"):
            _w = {k: float(v) for k, v in (weights or {}).items()}
            for k, v in _w.items():
                if k in df_or_arr.columns:
                    df_or_arr[k] = df_or_arr[k] * v
            return df_or_arr
        else:
            # array path
            if feature_names is None or not weights:
                return df_or_arr
            name_to_idx = {n: i for i, n in enumerate(feature_names)}
            out = df_or_arr.copy()
            for k, v in weights.items():
                if k in name_to_idx:
                    out[:, name_to_idx[k]] *= float(v)
            return out
    except Exception:
        return df_or_arr


# --- Optional PCA step (minimal patch) ---
USE_PCA = True  # set False to disable without restructuring any pipeline
PCA_VARIANCE = 0.95

_pca_fitted = None
try:
    from sklearn.decomposition import PCA as _PCA
except Exception:
    _PCA = None

def _maybe_fit_pca(X_train):
    global _pca_fitted
    if not USE_PCA or _PCA is None:
        return None
    _pca_fitted = _PCA(n_components=PCA_VARIANCE, svd_solver="full", random_state=42)
    _pca_fitted.fit(X_train)
    return _pca_fitted

def _maybe_apply_pca(X_any):
    if not USE_PCA or _PCA is None or _pca_fitted is None:
        return X_any
    return _pca_fitted.transform(X_any)


>>> Section 2: Imports + Reproducibility + Utilities
[STATUS] Utilities ready. N_THREADS= 7



## 3. Feature Engineering: PCA + Feature Weights

**Objective.** Consolidate Section 3 and 3b into a single, coherent feature engineering stage:
- Apply **PCA** to reduce collinearity and dimensionality while preserving ≥95% variance (configurable).
- Apply **per-feature weights** *after* preprocessing to attenuate or emphasise specific signals (e.g., assign a low weight to `ttl` rather than dropping it).

**Design choices (auditable):**
- PCA is post-`ColumnTransformer`, affecting the dense numeric feature matrix.
- `FeatureWeighter` multiplies transformed features by supplied weights without altering the pipeline API.
- Both components are guarded by idempotent patch blocks and config flags (`ENABLE_PCA`, `PCA_COMPONENTS`, `FEATURE_WEIGHTS`).

**Usage (examples):**
```python
FEATURE_WEIGHTS = {'ttl': 0.2, 'flow_duration': 1.5}  # names after preprocessing
ENABLE_PCA = True
PCA_COMPONENTS = 0.95  # keep 95% variance
```


In [24]:
# ======================================================
# Section 3 — Feature Engineering: PCA + Feature Weights
# Rationale:
#   • Merge former 3/3b into a single, auditable engineering stage.
#   • Apply analyst-defined weights on ORIGINAL features BEFORE PCA to shape variance.
#   • Use PCA to reduce collinearity and retain ≥95% variance by default.
# Outputs:
#   • Pipeline: preprocessor → restore_df → weighter → (optional) pca → clf
#   • Config knobs: FEATURE_WEIGHTS, ENABLE_PCA, PCA_COMPONENTS
#   • Audit prints: order, active weights, unmatched keys, PCA settings.
# ======================================================
print(">>> Section 3: Feature Engineering (weights before PCA)")

# ---- Configuration (edit as needed) ----
FEATURE_WEIGHTS = {'ttl': 0.2, 'flow_duration': 1.5}
ENABLE_PCA = True
PCA_COMPONENTS = 0.95

from typing import Optional, Dict, List

try:
    FeatureWeighter
except NameError:
    from sklearn.base import BaseEstimator, TransformerMixin
    import numpy as np, pandas as pd

    class FeatureWeighter(BaseEstimator, TransformerMixin):
        def __init__(self, weights: Optional[Dict]=None, feature_names: Optional[List]=None):
            self.weights = dict(weights or {})
            self.feature_names = list(feature_names) if feature_names is not None else None

        def fit(self, X, y=None):
            self.n_features_in_ = X.shape[1]
            return self

        def transform(self, X):
            import numpy as np, pandas as pd
            if isinstance(X, pd.DataFrame):
                df = X.copy()
                cols = df.columns.tolist()
            else:
                arr = np.asarray(X, dtype=object)
                n = arr.shape[1]
                cols = self.feature_names if self.feature_names is not None else [f"f{i}" for i in range(n)]
                df = pd.DataFrame(arr, columns=cols)
            W = pd.Series(1.0, index=cols, dtype=float)
            for k,v in self.weights.items():
                if isinstance(k,str) and k in W.index: W.loc[k]=float(v)
            for k,v in self.weights.items():
                if isinstance(k,int) and 0<=k<len(W): W.iloc[k]=float(v)
            num_cols = df.select_dtypes(include=[np.number]).columns
            if len(num_cols)>0:
                df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")
                df[num_cols] = df[num_cols].mul(W[num_cols].values, axis=1)
            return df

try:
    DataFrameRestorer
except NameError:
    from sklearn.base import BaseEstimator, TransformerMixin
    import pandas as pd
    class DataFrameRestorer(BaseEstimator, TransformerMixin):
        """Wrap ndarray outputs as DataFrame with provided column names for downstream name-aware steps.
        Robust to column-count mismatches (e.g., OHE category drift between subsets vs full fit).
        """
        def __init__(self, columns=None):
            self.columns = list(columns) if columns is not None else None

        def fit(self, X, y=None):
            # Only capture columns if not already provided; otherwise respect configured names.
            if self.columns is None and hasattr(X, "columns"):
                self.columns = list(X.columns)
            return self

        def transform(self, X):
            import numpy as np, pandas as pd
            # If upstream already outputs a DataFrame, prefer its columns (authoritative).
            if isinstance(X, pd.DataFrame):
                df = X.copy()
                if self.columns is not None and len(self.columns) == df.shape[1]:
                    df.columns = self.columns
                elif self.columns is not None and len(self.columns) != df.shape[1]:
                    # Shape mismatch: keep incoming names to avoid ValueError and log once.
                    try:
                        print(f"[Restorer] Column-count mismatch: incoming={df.shape[1]} vs stored={len(self.columns)}; keeping incoming names.")
                    except Exception:
                        pass
                return df

            # Otherwise, coerce to ndarray and synthesize column names if needed.
            arr = X.values if hasattr(X, "values") else np.asarray(X)
            n = arr.shape[1]
            if self.columns is not None and len(self.columns) == n:
                cols = self.columns
            else:
                cols = [f"f{i}" for i in range(n)]
                if self.columns is not None and len(self.columns) != n:
                    try:
                        print(f"[Restorer] Column-count mismatch: incoming={n} vs stored={len(self.columns)}; using generic names.")
                    except Exception:
                        pass
            return pd.DataFrame(arr, columns=cols)


from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import FunctionTransformer

if 'preprocessor' not in globals():
    print("[Warning] No ColumnTransformer `preprocessor` found. Using identity passthrough.")
    preprocessor=FunctionTransformer(lambda x:x)

if 'clf' not in globals():
    try:
        from lightgbm import LGBMClassifier
        clf=LGBMClassifier(random_state=42,n_estimators=300)
    except Exception:
        from sklearn.linear_model import SGDClassifier
        clf=SGDClassifier(random_state=42)

output_feature_names=None
try: output_feature_names=preprocessor.get_feature_names_out().tolist()
except Exception: pass

def _ensure_order(pipe_obj:Pipeline)->Pipeline:
    names=[n for n,_ in pipe_obj.steps]; steps=dict(pipe_obj.steps); ordered=[]
    for n in ["preprocessor","restore_df","weighter"]:
        if n in names: ordered.append((n,steps[n]))
    if "pca" in names: ordered.append(("pca",steps["pca"]))
    for n,s in pipe_obj.steps:
        if n not in [x[0] for x in ordered]: ordered.append((n,s))
    pipe_obj.steps=ordered; return pipe_obj

def _inject_pca(pipe_obj:Pipeline)->Pipeline:
    names=[n for n,_ in pipe_obj.steps]
    if ENABLE_PCA and "pca" not in names:
        insert_after = names.index("weighter") if "weighter" in names else (names.index("restore_df") if "restore_df" in names else 0)
        pipe_obj.steps.insert(insert_after+1,("pca",PCA(n_components=PCA_COMPONENTS,random_state=42)))
    return pipe_obj

if 'pipe' not in globals():
    pipe=Pipeline(steps=[("preprocessor",preprocessor),
                         ("restore_df",DataFrameRestorer(columns=output_feature_names)),
                         ("weighter",FeatureWeighter(weights=FEATURE_WEIGHTS,feature_names=output_feature_names)),
                         ("clf",clf)])
else:
    names=[n for n,_ in pipe.steps]
    if "restore_df" not in names: pipe.steps.insert(1,("restore_df",DataFrameRestorer(columns=output_feature_names)))
    names=[n for n,_ in pipe.steps]
    if "weighter" not in names:
        insert_at=names.index("restore_df")+1 if "restore_df" in names else (names.index("preprocessor")+1 if "preprocessor" in names else len(names))
        pipe.steps.insert(insert_at,("weighter",FeatureWeighter(weights=FEATURE_WEIGHTS,feature_names=output_feature_names)))
    pipe=_ensure_order(pipe)

pipe=_inject_pca(pipe)

_names=[n for n,_ in pipe.steps]
if "weighter" in _names and "pca" in _names:
    assert _names.index("weighter")<_names.index("pca"),"FeatureWeighter must precede PCA."

print("[Pipeline]"," -> ".join(n for n,_ in pipe.steps))
print(f"[Config] ENABLE_PCA={ENABLE_PCA} | PCA_COMPONENTS={PCA_COMPONENTS}")
try:
    non_unity={k:v for k,v in (FEATURE_WEIGHTS or {}).items() if float(v)!=1.0}
    print(f"[Weighter] Active={len(non_unity)>0} | Non-unity={non_unity}")
except Exception as e:
    print(f"[Weighter] Weight config warning: {e}")
try:
    if output_feature_names is not None:
        _unmatched=[k for k in (FEATURE_WEIGHTS or {}).keys() if isinstance(k,str) and k not in output_feature_names]
        if _unmatched: print(f"[Weighter] Unmatched weight keys (ignored): {_unmatched}")
except Exception as _e:
    print(f"[Weighter] Unmatched-keys check skipped: {_e}")


>>> Section 3: Feature Engineering (weights before PCA)
[Pipeline] preprocessor -> restore_df -> weighter -> pca -> clf
[Config] ENABLE_PCA=True | PCA_COMPONENTS=0.95
[Weighter] Active=True | Non-unity={'ttl': 0.2, 'flow_duration': 1.5}


In [25]:
# ======================================================
# Section 3b — Diagnostic: Sanity Fit & PCA Loadings
# Rationale:
#   • Verify pipeline executes end-to-end with weights before PCA.
#   • Provide transparent logs for auditability of config and PCA effects.
# Outputs:
#   • Printed pipeline topology.
#   • [Weighter] Active flag and non-unity weights.
#   • Top PCA component loadings mapped to original feature names.
# ======================================================
print(">>> Section 3b: Diagnostic — Sanity Fit & PCA Loadings")

from sklearn.utils import resample
from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np
import pandas as pd

# Heuristic: locate feature/label datasets defined earlier
_candidates = [
    ('X_train', 'y_train'),
    ('X', 'y'),
    ('Xt', 'yt'),
]
_found = None
for a,b in _candidates:
    if a in globals() and b in globals():
        _found = (a,b)
        break

if _found is None:
    print("[Diag] Could not locate (X, y) variables such as X_train/y_train. Skipping fit.")
else:
    Xa, ya = globals()[_found[0]], globals()[_found[1]]
    # Small stratified subset (up to 400 samples)
    try:
        if hasattr(ya, 'values'):
            y_arr = ya.values
        else:
            y_arr = ya
        sss = StratifiedShuffleSplit(n_splits=1, test_size=None, train_size=min(400, len(y_arr)), random_state=42)
        idx = next(sss.split(np.zeros(len(y_arr)), y_arr))[0]
        Xs = Xa.iloc[idx] if hasattr(Xa, 'iloc') else Xa[idx]
        ys = ya.iloc[idx] if hasattr(ya, 'iloc') else ya[idx]
    except Exception as e:
        print(f"[Diag] Stratified subset failed ({e}), falling back to head(400).")
        Xs = Xa[:400]
        ys = ya[:400]

    # Fit the pipeline
    assert 'pipe' in globals(), "Pipeline variable 'pipe' not found. Ensure Section 3 cell constructed 'pipe'."
    pipe.fit(Xs, ys)

    # Print pipeline
    print("[Pipeline]", " -> ".join(n for n,_ in pipe.steps))

    # Confirm weighter precedes PCA
    names = [n for n,_ in pipe.steps]
    if 'weighter' in names and 'pca' in names:
        print(f"[Order] 'weighter' @ {names.index('weighter')} < 'pca' @ {names.index('pca')} (OK)")
    else:
        print(f"[Order] Steps present: {names}")

    # Print active weights
    try:
        non_unity = {k:v for k,v in (FEATURE_WEIGHTS or {}).items() if float(v)!=1.0}
        print(f"[Weighter] Active={len(non_unity)>0} | Non-unity={non_unity}")
    except Exception as e:
        print(f"[Weighter] Check error: {e}")

    # PCA diagnostics
    try:
        pca_step = dict(pipe.steps).get('pca', None)
        if pca_step is None:
            print("[PCA] PCA disabled or not present.")
        else:
            pca = pca_step
            # Get original feature names post-restore_df / pre-PCA (weighter acts on these)
            try:
                feature_names = globals().get('output_feature_names', None)
                if feature_names is None:
                    # fallback to generic
                    feature_names = [f"f{i}" for i in range(pca.n_features_in_)]
            except Exception:
                feature_names = [f"f{i}" for i in range(getattr(pca, 'n_features_in_', 0) or len(pipe[:-1].transform(Xs)[0]))]

            evr = getattr(pca, 'explained_variance_ratio_', None)
            comps = getattr(pca, 'components_', None)
            if evr is None or comps is None:
                print("[PCA] Components/variance not available.")
            else:
                print(f"[PCA] n_components={getattr(pca, 'n_components_', getattr(pca, 'n_components', None))} | cum_var={evr.cumsum()[-1]:.3f}")
                # Top absolute loadings per first 3 PCs
                top_k = 10
                for pc_idx in range(min(3, comps.shape[0])):
                    weights = np.abs(comps[pc_idx])
                    top_idx = np.argsort(-weights)[:top_k]
                    top_features = [(feature_names[i] if i < len(feature_names) else f'f{i}', float(weights[i])) for i in top_idx]
                    print(f"[PCA] PC{pc_idx+1} top loadings:", top_features)
    except Exception as e:
        print(f"[PCA] Diagnostics error: {e}")

>>> Section 3b: Diagnostic — Sanity Fit & PCA Loadings
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.041528 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5353
[LightGBM] [Info] Number of data points in the train set: 400, number of used features: 40
[LightGBM] [Info] Start training from score -4.199705
[LightGBM] [Info] Start training from score -4.199705
[LightGBM] [Info] Start training from score -3.158251
[LightGBM] [Info] Start training from score -1.742969
[LightGBM] [Info] Start training from score -1.832581
[LightGBM] [Info] Start training from score -1.514128
[LightGBM] [Info] Start training from score -1.337504
[LightGBM] [Info] Start training from score -2.353878
[LightGBM] [Info] Start training from score -4.382027
[LightGBM] [Info] Start training from score -5.991465
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

In [26]:
# ======================================================
# Section 3c — Diagnostic: Pipeline & PCA Verification
# ======================================================
print(">>> Section 3c: Diagnostic — Pipeline & PCA Verification")

from sklearn.model_selection import StratifiedShuffleSplit
import numpy as np

# Try to locate dataset variables
_candidates = [
    ('X_train', 'y_train'),
    ('X', 'y'),
    ('Xt', 'yt'),
]
_found = None
for a, b in _candidates:
    if a in globals() and b in globals():
        _found = (a, b)
        break

if _found is None:
    print("[Diag] No dataset variables found (X_train/y_train, X/y). Skipping diagnostics.")
else:
    Xa, ya = globals()[_found[0]], globals()[_found[1]]

    # Sample a subset (≤400) with stratification if possible
    try:
        sss = StratifiedShuffleSplit(n_splits=1, train_size=min(400, len(ya)), random_state=42)
        idx = next(sss.split(np.zeros(len(ya)), ya))[0]
        Xs = Xa.iloc[idx] if hasattr(Xa, 'iloc') else Xa[idx]
        ys = ya.iloc[idx] if hasattr(ya, 'iloc') else ya[idx]
    except Exception as e:
        print("[Diag] Stratified sampling failed, falling back to first 400:", e)
        Xs, ys = Xa[:400], ya[:400]

    if 'pipe' not in globals():
        print("[Diag] No pipeline 'pipe' defined. Run Section 3 first.")
    else:
        # Pre-fit probe: check shapes after preprocessor
        try:
            Xt_probe = pipe.named_steps["preprocessor"].fit_transform(Xs)
            cols_probe = None
            if hasattr(Xt_probe, "shape"):
                print(f"[Probe] Preprocessor output shape on subset: {Xt_probe.shape}")
            if hasattr(pipe.named_steps["preprocessor"], "get_feature_names_out"):
                cols_probe = pipe.named_steps["preprocessor"].get_feature_names_out()
                print(f"[Probe] Feature name count from preprocessor: {len(cols_probe)}")
        except Exception as e:
            print(f"[Probe] Skipped preprocessor probe: {e}")

        # Fit pipeline on subset
        pipe.fit(Xs, ys)

        # Print effective pipeline order
        print("\n[Pipeline structure]", " -> ".join(n for n,_ in pipe.steps))

        names = [n for n,_ in pipe.steps]
        if 'weighter' in names and 'pca' in names:
            print(f"[Order] 'weighter' @ {names.index('weighter')} < 'pca' @ {names.index('pca')} (OK)")

        non_unity = {k:v for k,v in FEATURE_WEIGHTS.items() if float(v)!=1.0}
        print(f"[Weighter] Active={len(non_unity)>0} | Non-unity={non_unity}")

        pca = dict(pipe.steps).get('pca')
        if pca is None:
            print("[PCA] Disabled.")
        else:
            evr, comps = getattr(pca, 'explained_variance_ratio_', None), getattr(pca, 'components_', None)
            if evr is not None and comps is not None:
                print(f"[PCA] n_components={pca.n_components_} | cum_var={evr.cumsum()[-1]:.3f}")
                for pc_idx in range(min(3, comps.shape[0])):
                    weights = np.abs(comps[pc_idx])
                    top_idx = np.argsort(-weights)[:10]
                    # Use configured output_feature_names if set and non-None; otherwise synthesize generic names.
                    feat_names = globals().get('output_feature_names') or [f"f{i}" for i in range(len(weights))]
                    print(f"[PCA] PC{pc_idx+1} top loadings:",
                          [(feat_names[i], float(weights[i])) for i in top_idx])


>>> Section 3c: Diagnostic — Pipeline & PCA Verification
[Probe] Preprocessor output shape on subset: (400, 1514)
[Probe] Feature name count from preprocessor: 1514
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000990 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5353
[LightGBM] [Info] Number of data points in the train set: 400, number of used features: 40
[LightGBM] [Info] Start training from score -4.199705
[LightGBM] [Info] Start training from score -4.199705
[LightGBM] [Info] Start training from score -3.158251
[LightGBM] [Info] Start training from score -1.742969
[LightGBM] [Info] Start training from score -1.832581
[LightGBM] [Info] Start training from score -1.514128
[LightGBM] [Info] Start training from score -1.337504
[LightGBM] [Info] Start training from score -2.353878
[LightGBM] [Info] Start training from score -4.38202

## 4. Baseline: LightGBM (balanced)

Strong tabular baseline; reports **AUPRC**, **F1@τ**, **τ**, **TPR/TNR** and saves a stage manifest for later comparisons.


In [29]:
# ======================================================
# Section 4 — Baseline: LightGBM (adaptive objective)
# Rationale:
#   • Auto-detect labels: binary vs multiclass; set correct LightGBM objective.
#   • Robust target encoding with LabelEncoder for non-binary cases.
#   • Optional native categorical handling via `cat_cols` if defined in Section 2b.
# Outputs:
#   • Fit baseline LGBM; prints objective, classes, and class distribution.
#   • For binary: computes PR-AUC and threshold diagnostics later in the notebook.
# ======================================================
print(">>> Section 4: Baseline LightGBM (adaptive)")

from lightgbm import LGBMClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score, roc_auc_score
import numpy as np
import pandas as pd

# --- Preconditions / sanity checks ---
assert 'X_train' in globals() and 'y_train' in globals(), "X_train/y_train not found. Run Section 2a/2b."
assert len(y_train) > 0, "Empty y_train."
unique_classes = pd.Series(y_train).unique()
n_unique = len(unique_classes)
print(f"[Target] n_unique={n_unique} | classes={list(unique_classes)[:15]}")

if n_unique < 2:
    raise ValueError("y_train has a single class. Check your split/stratification and data filtering.")

# --- Decide objective & prepare y ---
# If dataset contains the canonical 'normal' vs attack taxonomy, default to binary: normal=0, else=1
use_binary_collapse = ('normal' in set(map(str, unique_classes))) and n_unique > 2

if n_unique == 2 or use_binary_collapse:
    # Binary case
    if use_binary_collapse:
        y_train_bin = (pd.Series(y_train).astype(str) != "normal").astype(int).values
        y_test_bin  = (pd.Series(y_test).astype(str)  != "normal").astype(int).values
        y_train_fit, y_test_eval = y_train_bin, y_test_bin
        print("[Labeling] Collapsed to binary: normal=0, non-normal=1")
    else:
        le = LabelEncoder()
        y_train_fit = le.fit_transform(y_train)
        y_test_eval = le.transform(y_test)
        print(f"[Labeling] Binary encoded with LabelEncoder: classes_={list(le.classes_)}")

    objective = "binary"
    extra_params = dict()
else:
    # Multiclass case
    le = LabelEncoder()
    y_train_fit = le.fit_transform(y_train)
    y_test_eval = le.transform(y_test)
    objective = "multiclass"
    extra_params = dict(num_class=len(le.classes_))
    print(f"[Labeling] Multiclass encoded: classes_={list(le.classes_)}")

# --- Baseline model ---
baseline = LGBMClassifier(
    objective=objective,
    n_estimators=400,
    learning_rate=0.05,
    max_depth=-1,
    subsample=0.9,
    colsample_bytree=0.9,
    class_weight="balanced",   # handles imbalance
    random_state=42,
    verbosity=-1,
    **extra_params
)

# --- Native categorical handling (if available from Section 2b) ---
cat_cols = globals().get("cat_cols", [])
fit_kwargs = {"categorical_feature": cat_cols} if cat_cols else {}
if cat_cols:
    print(f"[Categoricals] Using native categorical handling for {len(cat_cols)} columns.")

# --- Fit ---
baseline.fit(X_train, y_train_fit, **fit_kwargs)

# --- Evaluate (quick prints) ---
if objective == "binary":
    p_test = baseline.predict_proba(X_test)[:, 1]
    try:
        auprc = average_precision_score(y_test_eval, p_test)
        auc   = roc_auc_score(y_test_eval, p_test)
        print(f"[Eval] Binary AUPRC={auprc:.4f} | AUROC={auc:.4f} on test")
    except Exception as e:
        print(f"[Eval] Binary metrics skipped: {e}")
else:
    # Multiclass quick check: report per-class counts and model classes
    print(f"[Eval] Multiclass — classes: {extra_params.get('num_class')}, "
          f"train_dist={np.bincount(y_train_fit)} | test_dist={np.bincount(y_test_eval)}")

# Keep objects for later comparisons / thresholding blocks
baseline_model = baseline
baseline_objective = objective


>>> Section 4: Baseline LightGBM (adaptive)
[Target] n_unique=10 | classes=['fuzzers', 'generic', 'exploits', 'reconnaissance', 'normal', 'shellcode', 'dos', 'backdoor', 'analysis', 'worms']
[Labeling] Collapsed to binary: normal=0, non-normal=1
[Categoricals] Using native categorical handling for 1 columns.
[Eval] Binary AUPRC=0.9997 | AUROC=0.9993 on test


## 5. Hyperparameter Optimisation (HPO) — Overview

We optimise **PR-AUC** with:
- **5.1 Manual Grid (resumable)** — interpretable demo.
- **5.2 Randomized Search** — broader coverage under budget.
- **5.3 Bayesian Optimisation** — surrogate (GP) + EI, checkpoint.
- **5.4 Enhanced BO** — ask–tell, warm start, batch EI, patience, resume.


### 5.1 Manual Grid (resumable)
- interpretable demo, schema-safe resume

In [31]:
# ======================================================
# Section 5.1 — Cross-Validation (OOF PR-AUC) — patched
#   • Uses positional indexing (.iloc) to avoid KeyError
#   • Supports binary & multiclass (macro AUPRC)
#   • Preserves native-categorical fit if cat_cols provided
# ======================================================
print(">>> Section 5.1: CV (OOF PR-AUC) — patched indexing-safe")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.base import clone

# ---- Inputs / Preconditions
assert 'X_train' in globals() and 'y_train' in globals(), "Need X_train/y_train (run Section 2a/2b)."
est = globals().get('baseline_model', None)
if est is None:
    raise RuntimeError("baseline_model not found. Run Section 4 first to define the model.")

cat_cols = globals().get('cat_cols', [])

# Coerce to pandas objects with contiguous indices for safe iloc slicing
X_df = pd.DataFrame(X_train).reset_index(drop=True)
y_sr = pd.Series(y_train).reset_index(drop=True)

# Decide label mode (binary vs multiclass) consistent with Section 4
classes = y_sr.astype(str).unique()
use_binary_collapse = ('normal' in set(map(str, classes))) and len(classes) > 2
if use_binary_collapse:
    y_enc = (y_sr.astype(str) != "normal").astype(int)
    objective = "binary"
else:
    # if exactly 2 classes, still binary
    if y_sr.nunique() == 2:
        le = LabelEncoder()
        y_enc = pd.Series(le.fit_transform(y_sr))
        objective = "binary"
    else:
        le = LabelEncoder()
        y_enc = pd.Series(le.fit_transform(y_sr))
        objective = "multiclass"
        class_list = list(le.classes_)

print(f"[CV] objective={objective} | n_classes={y_sr.nunique()} | n_samples={len(y_sr)}")

# ---- CV setup
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

if objective == "binary":
    oof_pred = np.zeros(len(y_enc), dtype=float)
else:
    n_cls = y_sr.nunique()
    oof_pred = np.zeros((len(y_enc), n_cls), dtype=float)

fit_kwargs = {"categorical_feature": cat_cols} if cat_cols else {}

# ---- Manual CV loop to keep LightGBM kwargs
fold = 0
for tr_idx, va_idx in skf.split(X_df, y_enc):
    fold += 1
    X_tr, X_va = X_df.iloc[tr_idx, :], X_df.iloc[va_idx, :]
    y_tr, y_va = y_enc.iloc[tr_idx], y_enc.iloc[va_idx]

    model = clone(est)  # fresh clone each fold

    if objective == "binary":
        model.fit(X_tr, y_tr, **fit_kwargs)
        proba = model.predict_proba(X_va)[:, 1]
        oof_pred[va_idx] = proba
        ap = average_precision_score(y_va, proba)
        print(f"[Fold {fold}] AP={ap:.4f} (n_val={len(va_idx)})")
    else:
        model.fit(X_tr, y_tr, **fit_kwargs)
        proba = model.predict_proba(X_va)
        oof_pred[va_idx, :] = proba
        # macro AP for this fold
        Yb = label_binarize(y_va, classes=np.arange(proba.shape[1]))
        ap_macro = average_precision_score(Yb, proba, average="macro")
        print(f"[Fold {fold}] macro-AP={ap_macro:.4f} (n_val={len(va_idx)})")

# ---- Final OOF metrics
if objective == "binary":
    ap_oof = average_precision_score(y_enc, oof_pred)
    print(f"\n[OOF] Binary Average Precision (PR-AUC) over {n_splits} folds: {ap_oof:.4f}")
else:
    Yb_all = label_binarize(y_enc, classes=np.arange(oof_pred.shape[1]))
    ap_macro_oof = average_precision_score(Yb_all, oof_pred, average="macro")
    print(f"\n[OOF] Multiclass macro Average Precision over {n_splits} folds: {ap_macro_oof:.4f}")

# Keep for later sections
oof_predictions = oof_pred
oof_objective = objective


>>> Section 5.1: CV (OOF PR-AUC) — patched indexing-safe
[CV] objective=binary | n_classes=10 | n_samples=63904
[Fold 1] AP=0.9999 (n_val=12781)
[Fold 2] AP=0.9997 (n_val=12781)
[Fold 3] AP=0.9998 (n_val=12781)
[Fold 4] AP=1.0000 (n_val=12781)
[Fold 5] AP=0.9998 (n_val=12780)

[OOF] Binary Average Precision (PR-AUC) over 5 folds: 0.9998


### 5.2 Randomized Search (budget-controlled)

Covers a larger space with a finite budget. Scoring = `average_precision`.


In [33]:
# ======================================================
# Section 5.2 — Randomized Search (patched)
# ======================================================
print(">>> Section 5.2: Randomized Search (patched)")

import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import make_scorer, average_precision_score, f1_score
from lightgbm import LGBMClassifier

# ---- Expect X_train, y_train (from earlier sections)
assert 'X_train' in globals() and 'y_train' in globals(), "Need X_train and y_train in scope."
X_tr = X_train
y_sr = pd.Series(y_train).reset_index(drop=True)

# ---- Decide objective and encode y consistently
if any(str(c) == 'normal' for c in y_sr.unique()) and y_sr.nunique() > 2:
    # Collapse to binary: 1 = not-normal (attack), 0 = normal
    y_enc = (y_sr.astype(str) != 'normal').astype(int).to_numpy()
    lgb_objective = 'binary'
    num_class = None
elif y_sr.nunique() == 2:
    le = LabelEncoder().fit(y_sr)
    y_enc = le.transform(y_sr)
    lgb_objective = 'binary'
    num_class = None
else:
    le = LabelEncoder().fit(y_sr)
    y_enc = le.transform(y_sr)
    lgb_objective = 'multiclass'
    num_class = int(y_sr.nunique())

# ---- Scoring
def _ap_binary(y_true, y_pred_proba):
    if y_pred_proba.ndim == 2:
        y_pred_proba = y_pred_proba[:, 1]
    return average_precision_score(y_true, y_pred_proba)

if lgb_objective == 'binary':
    scoring = make_scorer(_ap_binary, needs_proba=True, greater_is_better=True)
else:
    scoring = make_scorer(f1_score, average="macro")

# ---- Base estimator configured to the objective
base = LGBMClassifier(
    objective=lgb_objective,
    num_class=num_class if lgb_objective == 'multiclass' else None,
    n_estimators=500,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# ---- Search space (objective-consistent)
param_dist = {
    "num_leaves": np.arange(16, 256),
    "max_depth": [-1] + list(range(4, 17)),
    "min_child_samples": np.arange(5, 101),
    "subsample": np.linspace(0.6, 1.0, 9),
    "colsample_bytree": np.linspace(0.6, 1.0, 9),
    "reg_alpha": np.linspace(0.0, 1.0, 21),
    "reg_lambda": np.linspace(0.0, 1.0, 21),
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

rs = RandomizedSearchCV(
    estimator=base,
    param_distributions=param_dist,
    n_iter=40,
    scoring=scoring,
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42,
    refit=True
)

# IMPORTANT: fit on the ENCODED targets
fit_kwargs = {'categorical_feature': cat_cols} if 'cat_cols' in globals() and cat_cols else {}
rs.fit(X_tr, y_enc, **fit_kwargs)

best_rs = rs.best_estimator_
print(f"[RS] Best CV score={rs.best_score_:.4f}")
print("[RS] Best params:", rs.best_params_)

# Expose for later sections
lgb_best = best_rs


>>> Section 5.2: Randomized Search (patched)
Fitting 3 folds for each of 40 candidates, totalling 120 fits


[RS] Best CV score=0.9999
[RS] Best params: {'subsample': 0.7, 'reg_lambda': 0.5, 'reg_alpha': 0.7000000000000001, 'num_leaves': 52, 'min_child_samples': 90, 'max_depth': 16, 'colsample_bytree': 1.0}


### 5.3 Bayesian Optimisation (BayesSearchCV) + Checkpoint

Surrogate (GP) + EI → fewer, better trials. A checkpoint allows resuming long searches safely.


In [38]:
# ================================
# Section 5.3 — BayesSearchCV (fixed)
# ================================
print(">>> Section 5.3: BO via BayesSearchCV (fixed)")

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import make_scorer, average_precision_score, f1_score
from lightgbm import LGBMClassifier
from skopt import BayesSearchCV
from skopt.space import Integer, Real

# -- Expect X_train, y_train in scope
assert 'X_train' in globals() and 'y_train' in globals(), "Need X_train and y_train."

X_tr = X_train
y_sr = pd.Series(y_train).reset_index(drop=True)

# -- Encode y and choose objective
# If more than 2 classes -> multiclass; otherwise binary
unique_vals = pd.unique(y_sr)
if len(unique_vals) > 2:
    le = LabelEncoder().fit(y_sr)
    y_enc = le.transform(y_sr)
    lgb_objective = 'multiclass'
else:
    # Ensure {0,1} encoding
    if y_sr.dtype.kind in "biufc":
        # numeric → map smallest to 0, largest to 1
        le = LabelEncoder().fit(y_sr)
        y_enc = le.transform(y_sr)
    else:
        # string labels (e.g., 'normal', 'attack' etc.)
        le = LabelEncoder().fit(y_sr)
        y_enc = le.transform(y_sr)
    lgb_objective = 'binary'

# -- Scorer aligned with objective
def _ap_binary(y_true, y_pred):
    # y_pred may be proba matrix or vector
    y_pred = y_pred[:, 1] if y_pred.ndim == 2 else y_pred
    return average_precision_score(y_true, y_pred)

scoring = (
    make_scorer(_ap_binary, needs_proba=True, greater_is_better=True)
    if lgb_objective == 'binary'
    else make_scorer(f1_score, average="macro")
)

# -- Base estimator
# IMPORTANT: do NOT set num_class explicitly; LightGBM will infer it
base = LGBMClassifier(
    objective=lgb_objective,
    n_estimators=600,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# -- Search space (no 'objective', no 'num_class' here)
search_spaces = {
    "num_leaves": Integer(16, 256),
    "max_depth": Integer(-1, 16),           # -1 is unlimited
    "min_child_samples": Integer(5, 100),
    "subsample": Real(0.6, 1.0),
    "colsample_bytree": Real(0.6, 1.0),
    "reg_alpha": Real(0.0, 1.0),
    "reg_lambda": Real(0.0, 1.0),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

opt = BayesSearchCV(
    estimator=base,
    search_spaces=search_spaces,
    n_iter=30,                # adjust as you like
    cv=cv,
    n_jobs=-1,
    scoring=scoring,
    random_state=42,
    verbose=1,
    refit=True,
    error_score='raise'       # fail fast if anything goes wrong
)

# Pass categorical columns if you have them (names or indices). Otherwise leave empty.
fit_kwargs = {'categorical_feature': cat_cols} if 'cat_cols' in globals() and cat_cols else {}
# --- extra safety guards (optional) ---

opt.fit(X_tr, y_enc, **fit_kwargs)

from sklearn.metrics import average_precision_score, f1_score

# Reuse the same label encoder for test y so classes line up
y_test_enc = le.transform(pd.Series(y_test))

p_test = lgb_bo.predict_proba(X_test)

if lgb_objective == 'binary':
    ap_test = average_precision_score(y_test_enc, p_test[:, 1])
    print(f"[BO] Test AP = {ap_test:.4f}")
else:
    y_pred = p_test.argmax(axis=1)
    f1m = f1_score(y_test_enc, y_pred, average='macro')
    print(f"[BO] Test macro-F1 = {f1m:.4f}")

print(f"[BO] Best CV score={opt.best_score_:.4f} | Best params={opt.best_params_}")
lgb_bo = opt.best_estimator_


>>> Section 5.3: BO via BayesSearchCV (fixed)
Fitting 5 folds for each of 1 candidates, totalling 5 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 5.101875 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 320726
[LightGBM] [Info] Number of data points in the train set: 51124, number of used features: 1464
[LightGBM] [Info] Start training from score -4.193025
[LightGBM] [Info] Start training from score -4.166186
[LightGBM] [Info] Start training from score -3.157225
[LightGBM] [Info] Start training from score -1.742154
[LightGBM] [Info] Start training from score -1.837095
[LightGBM] [Info] Start training from score -1.513797
[LightGBM] [Info] Start training from score -1.336019
[LightGBM] [Info] Start training from score -2.357339
[LightGBM] [Info] Start training from score -4.296660
[LightGBM] [Info] Start training from score -6.747665
[LightGBM] [Warning] No further splits with positive gain, bes

Exception ignored on calling ctypes callback function: <function _log_callback at 0x12aeb4cc0>
Traceback (most recent call last):
  File "/opt/miniconda3/envs/py312/lib/python3.12/site-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


No further splits with positive gain, best gain: -inf


Exception ignored on calling ctypes callback function: <function _log_callback at 0x1251e4cc0>
Traceback (most recent call last):
  File "/opt/miniconda3/envs/py312/lib/python3.12/site-packages/lightgbm/basic.py", line 287, in _log_callback
    def _log_callback(msg: bytes) -> None:
    
KeyboardInterrupt: 


No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


KeyboardInterrupt: 

### 5.4 Enhanced BO (ask–tell, warm start, batch EI, patience, resume)

Warm start from earlier bests; propose batches; early stop on **patience**; resume via CSV + pickle.


In [ ]:
# ======================================================
# Cell 15 — 5.4 Enhanced BO (ask–tell)
# Features:
#   • Warm-start from previous staged bests (ManualGrid/RS/BO).
#   • Batch EI proposals (parallel CV evaluations).
#   • Patience-based early stopping (prevents over-search).
#   • Resume via optimizer.pkl + results.csv.
# ======================================================
print(">>> Section 5.4: Enhanced BO (ask–tell)")

ben_dir = stage_dir('bo_enhanced')
opt_pkl = ben_dir / 'optimizer.pkl'   # skopt.Optimizer state (pickle)
res_csv = ben_dir / 'results.csv'     # history table for plots/analysis

# Search space mirrors BayesSearchCV’s; names must match vec_to_params below
space = [
  Integer(31, 255, name='num_leaves'),
  Integer(2, 16, name='max_depth'),
  Integer(10, 200, name='min_child_samples'),
  Real(0.6, 1.0, name='subsample'),
  Real(0.6, 1.0, name='colsample_bytree'),
  Real(1e-3, 2e-1, prior='log-uniform', name='learning_rate'),
  Real(1e-3, 10.0, prior='log-uniform', name='reg_lambda')
]

# ---- Warm start from staged champions --------------------------------------
X0, y0 = [], []   # X0: hyperparameters as vectors; y0: objective values (-AP)
for stg in ['manual_grid', 'random_search', 'bo_lgb']:
    m = load_stage(stg)
    if not m: 
        continue
    p = m.get('best_params') or m.get('params')
    if not p: 
        continue
    x = [
        int(p.get('num_leaves',63)),
        int(p.get('max_depth',6)),
        int(p.get('min_child_samples',20)),
        float(p.get('subsample',0.8)),
        float(p.get('colsample_bytree',0.8)),
        float(p.get('learning_rate',0.1)),
        float(p.get('reg_lambda',1.0)),
    ]
    ap = m.get('metrics',{}).get('AUPRC')
    if ap is not None:
        X0.append(x); y0.append(-float(ap))  # skopt minimizes; we maximize AP → minimize -AP

# ---- Initialize or resume Optimizer ----------------------------------------
if opt_pkl.exists():
    # Resume optimizer from checkpoint to continue search
    opt = pickle.load(open(opt_pkl,'rb'))
    print("[EBO] Loaded optimizer from checkpoint.")
else:
    opt = Optimizer(dimensions=space, base_estimator='GP', acq_func='EI', random_state=RANDOM_STATE)
    if X0:
        opt.tell(X0, y0)  # inject prior bests to guide the surrogate
        print(f"[EBO] Warm-start with {len(X0)} prior points.")

def cv_ap_eval(params: dict) -> float:
    """Return mean 3-fold CV AP for given LightGBM params."""
    clf = LGBMClassifier(
        objective='binary', n_estimators=500, random_state=RANDOM_STATE,
        n_jobs=-1, class_weight='balanced', verbosity=-1, **params
    )
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    aps = []
    for tr, va in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[tr], X_train.iloc[va]
        y_tr, y_va = y_train[tr], y_train[va]
        fit_kwargs = {'categorical_feature': cat_cols} if cat_cols else {}
        clf.fit(X_tr, y_tr, **fit_kwargs)
        p = clf.predict_proba(X_va)[:, 1]
        aps.append(average_precision_score(y_va, p))
    return float(np.mean(aps))

def vec_to_params(v):
    """Vector → dict(params) mapping; keep order in sync with 'space' above."""
    return {
        'num_leaves': int(v[0]),
        'max_depth': int(v[1]),
        'min_child_samples': int(v[2]),
        'subsample': float(v[3]),
        'colsample_bytree': float(v[4]),
        'learning_rate': float(v[5]),
        'reg_lambda': float(v[6])
    }

# ---- Load history or start fresh -------------------------------------------
hist = pd.read_csv(res_csv) if res_csv.exists() else pd.DataFrame(columns=[
    'num_leaves','max_depth','min_child_samples','subsample','colsample_bytree','learning_rate','reg_lambda','AP_cv'
])

# Search controls (keep modest for demo; raise for production)
N_ITERS, BATCH, PATIENCE = 24, min(4, N_THREADS), 6
best_cv = float(hist['AP_cv'].max()) if not hist.empty else -np.inf
no_imp = 0

# ---- Main ask–tell loop -----------------------------------------------------
for step in range(N_ITERS):
    # Ask for next batch of proposals from the surrogate
    Xb = opt.ask(n_points=BATCH)
    Pb = [vec_to_params(v) for v in Xb]
    print(f"[EBO] Step {step+1}/{N_ITERS} proposing {len(Pb)} points...")
    # Evaluate proposals in parallel
    scores = Parallel(n_jobs=BATCH)(delayed(cv_ap_eval)(p) for p in Pb)
    # Tell the optimizer the outcomes (remember we minimize -AP)
    opt.tell(Xb, [-s for s in scores])
    # Append to history & checkpoint both optimizer and CSV
    rows = [dict(**p, AP_cv=s) for p, s in zip(Pb, scores)]
    hist = pd.concat([hist, pd.DataFrame(rows)], ignore_index=True)
    hist.to_csv(res_csv, index=False)
    pickle.dump(opt, open(opt_pkl,'wb'))
    # Patience logic: stop if no improvement
    b = float(max(scores))
    if b > best_cv + 1e-6:
        best_cv, no_imp = b, 0
    else:
        no_imp += 1
    print(f"[EBO] step={step+1} | batch_best={b:.4f} | best_cv={best_cv:.4f} | patience {no_imp}/{PATIENCE}")
    if no_imp >= PATIENCE:
        print("[EBO] Early stopping."); break

# ---- Finalize: pick best params; refit on full train; evaluate on test -----
if not hist.empty:
    best_row = hist.sort_values('AP_cv', ascending=False).iloc[0].to_dict()
    key_list = ['num_leaves','max_depth','min_child_samples','subsample','colsample_bytree','learning_rate','reg_lambda']
    best_params = {k: best_row[k] for k in key_list}
    print(f"[EBO] Best params={best_params} | Best CV AP={best_row['AP_cv']:.4f}")

    lgb_enh = LGBMClassifier(
        objective='binary', n_estimators=500, random_state=RANDOM_STATE,
        n_jobs=-1, class_weight='balanced', verbosity=-1, **best_params
    )
    fit_kwargs = {'categorical_feature': cat_cols} if cat_cols else {}
    lgb_enh.fit(X_train, y_train, **fit_kwargs)
    p = lgb_enh.predict_proba(X_test)[:, 1]
    ap = average_precision_score(y_test, p)
    f1b, tau = sweep_f1(y_test, p)
    tpr, tnr = tpr_tnr_at_tau(y_test, p, tau)
    print(f"[EBO] Test AP={ap:.4f} | F1@τ={f1b:.4f} | τ={tau:.2f}")

    save_stage('bo_enhanced', {
      'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
      'stage': 'bo_enhanced',
      'model_name': 'LightGBM_BO_Enhanced',
      'dataset': DATA_PATH,
      'seed': RANDOM_STATE,
      'metrics': {'AUPRC': float(ap), 'F1': float(f1b), 'tau': float(tau), 'TPR': float(tpr), 'TNR': float(tnr)},
      'params': best_params
    }, results=hist, model=lgb_enh)
else:
    print("[EBO] No evaluations recorded.")


## 6. 1D-CNN Benchmark

Treat each sample as a **length-d 1D signal** (d = #features).  
Small architecture for speed; predictions saved to `staging/cnn1d/`.


In [ ]:
# ======================================================
# Cell 17 — 1D-CNN Baseline
# Idea:
#   • Treat each tabular sample as a 1-D signal of length d (#features).
#   • Minimal CNN acts as a neural baseline against tree-based models.
# Repro:
#   • Save predictions to staging/cnn1d/preds.npy so ensembles can include it.
# ======================================================
print(">>> Section 6: 1D-CNN Benchmark")
try:
    import tensorflow as tf
    from tensorflow.keras import Sequential
    from tensorflow.keras.layers import Input, Conv1D, GlobalMaxPooling1D, Dense, Dropout, BatchNormalization
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    try:
        from tensorflow.keras.callbacks import BackupAndRestore   # resume support (when available)
        _has_backup = True
    except Exception:
        _has_backup = False
    TF_OK = True
except Exception as e:
    print("[CNN1D] TensorFlow unavailable:", e)
    TF_OK = False

def to_seq(Xtr: pd.DataFrame, Xte: pd.DataFrame):
    """
    Convert DataFrame → numeric arrays → (N, D, 1) tensors.
    - Category columns → integer codes (stable due to earlier alignment).
    - Booleans → compact ints; objects → categories.
    """
    A, B = Xtr.copy(), Xte.copy()
    for c in A.columns:
        if str(A[c].dtype) == 'category':
            A[c] = A[c].cat.codes; B[c] = B[c].cat.codes
        elif A[c].dtype == bool:
            A[c] = A[c].astype(np.int8); B[c] = B[c].astype(np.int8)
        elif A[c].dtype == object:
            A[c] = A[c].astype('category').cat.codes
            B[c] = B[c].astype('category').cat.codes
    A = A.fillna(0).astype(np.float32).values
    B = B.fillna(0).astype(np.float32).values
    return A.reshape((A.shape[0], A.shape[1], 1)), B.reshape((B.shape[0], B.shape[1], 1))

if TF_OK:
    Xtr3, Xte3 = to_seq(X_train, X_test)
    L = Xtr3.shape[1]  # number of features = sequence length

    # Lightweight CNN: two conv blocks + global max pool + small MLP head
    model = Sequential([
        Input(shape=(L,1)),
        Conv1D(32, 3, activation='relu', padding='same'),
        BatchNormalization(),
        Conv1D(32, 3, activation='relu', padding='same'),
        GlobalMaxPooling1D(),
        Dropout(0.2),                  # implicit regularization
        Dense(64, activation='relu'),
        Dense(1, activation='sigmoid') # output = P(y=1)
    ])
    model.compile(optimizer=Adam(1e-3), loss='binary_crossentropy',
                  metrics=[tf.keras.metrics.AUC(curve='PR', name='AUPRC')])

    # Early stopping: maximize validation PR-AUC; LR scheduling for stability
    callbacks=[EarlyStopping(monitor='val_AUPRC', mode='max', patience=5, restore_best_weights=True),
               ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5)]
    cnn_stage = stage_dir('cnn1d')
    if _has_backup:
        callbacks.append(BackupAndRestore(backup_dir=str(cnn_stage/'tf_backup')))

    print('[CNN1D] Training...')
    hist = model.fit(Xtr3, y_train, validation_split=0.2, epochs=30, batch_size=256,
                     callbacks=callbacks, verbose=1)

    # Evaluate and persist predictions for later ensembles/plots
    p = model.predict(Xte3, batch_size=1024, verbose=0).ravel()
    ap = average_precision_score(y_test, p)
    f1b, tau = sweep_f1(y_test, p)
    tpr, tnr = tpr_tnr_at_tau(y_test, p, tau)

    np.save(cnn_stage/'preds.npy', p)                 # for ensembles
    model.save(cnn_stage/'model.keras', include_optimizer=False)  # model archive

    print(f"[CNN1D] Test AP={ap:.4f} | F1@τ={f1b:.4f} | τ={tau:.2f}")
    save_stage('cnn1d', {
      'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
      'stage': 'cnn1d',
      'model_name': 'CNN1D_Minimal',
      'dataset': DATA_PATH,
      'seed': RANDOM_STATE,
      'metrics': {'AUPRC': float(ap), 'F1': float(f1b), 'tau': float(tau), 'TPR': float(tpr), 'TNR': float(tnr)}
    })
else:
    print('[CNN1D] Skipped (TF not available).')


# --- Minimal CNN grid search (filters, kernel_size) with residual connection ---
GRID_FILTERS = [16, 32]
GRID_KERNELS = [3, 5]

best_hist, best_cfg = None, None
best_score = -1.0

def _build_simple_1dcnn(input_shape, num_classes, filters=32, kernel_size=3):
    import tensorflow as tf
    from tensorflow.keras import layers, models

    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    skip = x  # for residual
    x = layers.Conv1D(filters, kernel_size, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, skip])  # residual
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs=inp, outputs=out)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


best_hist, best_cfg, best_score = None, None, -1.0
for _f in GRID_FILTERS:
    for _k in GRID_KERNELS:
        _model = _build_simple_1dcnn(input_shape=X_train.shape[1:], num_classes=len(set(y_train)), filters=_f, kernel_size=_k)
        _h = _model.fit(X_train, y_train, validation_data=(X_valid, y_valid), epochs=20, batch_size=128, verbose=0, callbacks=[EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True), ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=2, verbose=0)])
        _val_acc = max(_h.history.get("val_accuracy", [0.0]))
        if _val_acc > best_score:
            best_score, best_hist, best_cfg = _val_acc, _h, {"filters": _f, "kernel_size": _k}
# keep last model as best for minimal intrusion; you may rebuild with best_cfg if desired


## 7. Ensembles

- **RF bagging** baseline.
- **Soft voting** across staged predictors.
- **Stacking** with logistic meta-learner (no test leakage).


In [ ]:
# ======================================================
# Cell 19 — Ensembles: bagging + soft voting + stacking
# Motivation (Module 10):
#   • Bagging reduces variance (RF).
#   • Voting blends calibrated probabilities.
#   • Stacking learns a meta-combiner while preventing test leakage.
# ======================================================
print(">>> Section 7: Ensembles — bagging + voting + stacking")

# Minimal fix: encode categorical features
from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[cat_cols] = oe.fit_transform(X_train[cat_cols])
X_test[cat_cols] = oe.transform(X_test[cat_cols])

from sklearn.ensemble import RandomForestClassifier
import joblib

# --- Bagging baseline (RandomForest) ----------------------------------------
rf = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced_subsample',   # each tree sees balanced bootstrap
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf.fit(X_train, y_train)
p = rf.predict_proba(X_test)[:, 1]
ap = average_precision_score(y_test, p)
f1b, tau = sweep_f1(y_test, p)
print(f"[RF] Test AP={ap:.4f} | F1@τ={f1b:.4f} | τ={tau:.2f}")
save_stage('ensemble_rf', {
  'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
  'stage': 'ensemble_rf',
  'model_name': 'RandomForest_Bagging',
  'dataset': DATA_PATH,
  'seed': RANDOM_STATE,
  'metrics': {'AUPRC': float(ap), 'F1': float(f1b), 'tau': float(tau)}
}, model=rf)

# --- Soft voting over staged predictors -------------------------------------
def load_preds_for(stage_name: str):
    """
    Try to load predictions for a staged model on X_test:
      • tabular models: load model.joblib and run predict_proba.
      • CNN1D: predictions saved earlier to preds.npy.
    """
    pth = Path('staging')/stage_name/'model.joblib'
    if pth.exists():
        mdl = joblib.load(pth)
        try:
            return mdl.predict_proba(X_test)[:, 1]
        except Exception:
            return None
    if stage_name == 'cnn1d':
        npy = Path('staging')/'cnn1d'/'preds.npy'
        if npy.exists():
            arr = np.load(npy)
            return arr if len(arr) == len(y_test) else None
    return None

members = ['bo_enhanced','bo_lgb','random_search','manual_grid','ensemble_rf','baseline','cnn1d']
preds = {m: load_preds_for(m) for m in members}
preds = {k: v for k, v in preds.items() if v is not None}
print(f"[VOTE] Using members: {list(preds.keys())}")

if len(preds) >= 2:
    # Uniform weights are a strong baseline;
    # could be replaced with validation-based weights if desired.
    W = np.ones(len(preds)) / len(preds)
    P = np.stack(list(preds.values()), axis=1)
    p_vote = (P * W).sum(axis=1)

    ap = average_precision_score(y_test, p_vote)
    f1b, tau = sweep_f1(y_test, p_vote)
    print(f"[VOTE] Test AP={ap:.4f} | F1@τ={f1b:.4f} | τ={tau:.2f}")

    save_stage('ensemble_soft', {
      'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
      'stage': 'ensemble_soft',
      'model_name': 'Ensemble_SoftVoting',
      'dataset': DATA_PATH,
      'seed': RANDOM_STATE,
      'members': list(preds.keys()),
      'metrics': {'AUPRC': float(ap), 'F1': float(f1b), 'tau': float(tau)}
    })
else:
    print('[VOTE] Need ≥2 members with predictions, skipping.')

# --- Stacking with logistic meta-learner ------------------------------------
# Avoid test leakage: split the training set (A/B) to create meta-features on B.
from sklearn.linear_model import LogisticRegression
XA, XB, yA, yB = train_test_split(X_train, y_train, test_size=0.2,
                                  random_state=RANDOM_STATE, stratify=y_train)

# Pull parameterized LightGBM champions from staged searches
specs = []
for s in ['bo_enhanced','bo_lgb','random_search','manual_grid']:
    m = load_stage(s)
    if not m: 
        continue
    p = m.get('best_params') or m.get('params')
    if p: 
        specs.append((s, p))

if specs:
    ZB, ZT, names = [], [], []
    for name, params in specs:
        clf = LGBMClassifier(
            objective='binary', n_estimators=500, random_state=RANDOM_STATE,
            n_jobs=-1, class_weight='balanced', verbosity=-1, **params
        )
        fit_kwargs = {'categorical_feature': cat_cols} if cat_cols else {}
        clf.fit(XA, yA, **fit_kwargs)
        # Meta-train features: predictions on XB
        ZB.append(clf.predict_proba(XB)[:, 1])
        # Meta-test features: predictions on X_test (no leakage)
        ZT.append(clf.predict_proba(X_test)[:, 1])
        names.append(name)
    ZB = np.stack(ZB, axis=1)
    ZT = np.stack(ZT, axis=1)

    # Logistic meta-learner is simple & interpretable
    meta = LogisticRegression(class_weight='balanced', max_iter=300, random_state=RANDOM_STATE)
    meta.fit(ZB, yB)
    p_stack = meta.predict_proba(ZT)[:, 1]

    ap = average_precision_score(y_test, p_stack)
    f1b, tau = sweep_f1(y_test, p_stack)
    print(f"[STACK] Test AP={ap:.4f} | F1@τ={f1b:.4f} | τ={tau:.2f} | members={names}")

    save_stage('ensemble_stack', {
      'timestamp': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
      'stage': 'ensemble_stack',
      'model_name': 'Ensemble_BlendedStack',
      'dataset': DATA_PATH,
      'seed': RANDOM_STATE,
      'members': names,
      'meta': 'LogisticRegression',
      'metrics': {'AUPRC': float(ap), 'F1': float(f1b), 'tau': float(tau)}
    })
else:
    print('[STACK] No staged LightGBM params to stack; skipping.')


## 8. Analysis & Visual Justification

- Metrics table across staged models.
- Enhanced BO convergence (best CV AP vs evaluation).
- Hyperparameter response (AP vs single parameter).
- PR curves (Module 3).


In [ ]:
# ======================================================
# Cell 21 — Analysis & Visual Justification
# Includes:
#   • Metrics table across staged models.
#   • Enhanced BO convergence plot (best-so-far CV AP).
#   • Param-response scatter (AP vs hyperparameter).
#   • PR curves comparing champions (Module 3).
# Note:
#   • Plots do not set custom colors per coursework rules; default Matplotlib.
# ======================================================
print(">>> Section 8: Analysis & Visuals")
import matplotlib.pyplot as plt

# ---- 1) Metrics table (Champion comparison) --------------------------------
stages = ['baseline','manual_grid','random_search','bo_lgb','bo_enhanced','ensemble_rf','ensemble_soft','ensemble_stack','cnn1d']
rows = []
for s in stages:
    m = load_stage(s)
    if m and 'metrics' in m:
        rows.append({'stage': s, **m['metrics']})
metrics_df = pd.DataFrame(rows)
if not metrics_df.empty:
    display(metrics_df.sort_values(['AUPRC','F1'], ascending=[False, False]).reset_index(drop=True))
    print('[ANALYSIS] Metrics table displayed.')
else:
    print('[ANALYSIS] No staged metrics yet.')

# ---- 2) Enhanced BO convergence plot ---------------------------------------
ben_hist = stage_dir('bo_enhanced') / 'results.csv'
plt.figure()
if ben_hist.exists():
    h = pd.read_csv(ben_hist)
    if not h.empty and 'AP_cv' in h.columns:
        best_so_far = h['AP_cv'].cummax().values
        plt.plot(range(1, len(best_so_far)+1), best_so_far, marker='o')
        plt.xlabel('Enhanced BO evaluations'); plt.ylabel('Best CV AP so far')
        plt.title('Enhanced BO Convergence')
        plt.grid(True, linewidth=0.3)
    else:
        plt.text(0.5,0.5,'No AP_cv history.',ha='center')
else:
    plt.text(0.5,0.5,'No bo_enhanced/results.csv.',ha='center')
plt.savefig('images/plot_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png')
plt.show()

# ---- 3) Hyperparameter response (example: num_leaves) ----------------------
if ben_hist.exists():
    h = pd.read_csv(ben_hist)
    if {'num_leaves','AP_cv'} <= set(h.columns):
        plt.figure()
        plt.scatter(h['num_leaves'], h['AP_cv'], s=12)
        plt.xlabel('num_leaves'); plt.ylabel('CV AP'); plt.title('num_leaves vs CV AP')
        plt.grid(True, linewidth=0.3); plt.savefig('images/plot_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png')
plt.show()

# ---- 4) Precision–Recall curves -------------------------------------------
from sklearn.metrics import precision_recall_curve
import joblib

def pr_arrays(y_true, p):
    P, R, T = precision_recall_curve(y_true, p)
    ap = average_precision_score(y_true, p)
    return P, R, ap

# Gather prediction functions for staged models (re-using saved artifacts)
curves = {}
# Baseline predictions captured at training time
if 'p_base' in globals():
    curves['baseline'] = p_base

for s in ['bo_enhanced','bo_lgb','random_search','manual_grid','ensemble_rf','ensemble_soft','ensemble_stack']:
    mp = stage_dir(s) / 'model.joblib'
    if mp.exists():
        try:
            mdl = joblib.load(mp)
            curves[s] = mdl.predict_proba(X_test)[:, 1]
        except Exception:
            pass

# CNN1D saved proba predictions
pn = stage_dir('cnn1d') / 'preds.npy'
if pn.exists():
    curves['cnn1d'] = np.load(pn)

plt.figure()
for name, pr in list(curves.items()):
    if pr is None:
        continue
    P, R, AP = pr_arrays(y_test, pr)
    plt.plot(R, P, linewidth=1, label=f"{name} (AP={AP:.3f})")
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision–Recall Curves')
plt.grid(True, linewidth=0.3); plt.legend(); plt.savefig('images/plot_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png')
plt.show()
print('[ANALYSIS] Plots generated.')


# --- Flexible Feature Weighting (minimal patch) ---
# You can assign weights to any subset of features; unspecified features default to 1.0.
FEATURE_WEIGHTS = {  # e.g., {"ttl": 0.1, "payload_byte_1": 0.5}
    # "ttl": 0.1,
}
def _apply_feature_weights(df_or_arr, feature_names=None, weights=FEATURE_WEIGHTS):
    """
    Minimal intrusive scaler: multiply specified columns by given weights.
    - If df_or_arr is a pandas DataFrame: use column names directly.
    - If it's a NumPy array: use feature_names to locate columns.
    """
    try:
        import pandas as _pd
        if hasattr(df_or_arr, "columns"):
            _w = {k: float(v) for k, v in (weights or {}).items()}
            for k, v in _w.items():
                if k in df_or_arr.columns:
                    df_or_arr[k] = df_or_arr[k] * v
            return df_or_arr
        else:
            # array path
            if feature_names is None or not weights:
                return df_or_arr
            name_to_idx = {n: i for i, n in enumerate(feature_names)}
            out = df_or_arr.copy()
            for k, v in weights.items():
                if k in name_to_idx:
                    out[:, name_to_idx[k]] *= float(v)
            return out
    except Exception:
        return df_or_arr


# --- Optional PCA step (minimal patch) ---
USE_PCA = True  # set False to disable without restructuring any pipeline
PCA_VARIANCE = 0.95

_pca_fitted = None
try:
    from sklearn.decomposition import PCA as _PCA
except Exception:
    _PCA = None

def _maybe_fit_pca(X_train):
    global _pca_fitted
    if not USE_PCA or _PCA is None:
        return None
    _pca_fitted = _PCA(n_components=PCA_VARIANCE, svd_solver="full", random_state=42)
    _pca_fitted.fit(X_train)
    return _pca_fitted

def _maybe_apply_pca(X_any):
    if not USE_PCA or _PCA is None or _pca_fitted is None:
        return X_any
    return _pca_fitted.transform(X_any)


In [ ]:
# ======================================================
# Cell 8b — SHAP: Fast (cached & sampled, no retrain)
# What:
#   • Load champion model from staging (no refit).
#   • Compute SHAP on a SMALL stratified sample (≈2k rows) with fast tree explainer.
#   • Cache SHAP arrays to avoid recomputation; then save & show 3 plots.
# Why (speed):
#   • SHAP on full X_train is O(Trees × Samples) and slow.
#   • Sampling + caching + tree_path_dependent + approximate=True cuts runtime drastically.
# ======================================================
print(">>> Cell 8b: SHAP — fast cached & sampled (no retrain)")

# ---- Imports
import os, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import joblib
import shap

# ---- Preconditions / helpers
assert 'X_train' in globals() and 'y_train' in globals(), \
    "Please run earlier training cells first to define X_train, y_train."

def stage_dir(stage): 
    return Path('staging') / stage

def load_stage(stage):
    p = stage_dir(stage) / "manifest.json"
    return json.loads(p.read_text()) if p.exists() else {}

# ---- Pick champion (by highest AUPRC in manifests)
champ_sources = ["bo_enhanced", "bo_lgb", "random_search", "manual_grid"]
best = None
for s in champ_sources:
    m = load_stage(s)
    if m and m.get("metrics", {}).get("AUPRC") is not None:
        if (best is None) or (m["metrics"]["AUPRC"] > best["metrics"]["AUPRC"]):
            best = m | {"stage": s}
assert best is not None, "No staged champions found under staging/<stage>/manifest.json."

champ_stage = best["stage"]
model_path = stage_dir(champ_stage) / "model.joblib"
assert model_path.exists(), f"Missing model artifact: {model_path}"

print(f"[SHAP] Champion: {champ_stage} | model: {model_path.name}")

# ---- Load fitted model (NO retrain)
model = joblib.load(model_path)

# ---- Create output/cache dirs
out_dir = Path("staging")
out_dir.mkdir(parents=True, exist_ok=True)
cache_path = out_dir / f"shap_cache_{champ_stage}.joblib"

# ---- Build a SMALL sample for speed (stratified if possible)
rng = np.random.RandomState(1337)
N = len(X_train)
sample_size = min(2000, N)  # tune down for more speed if needed
yv = y_train.values if hasattr(y_train, 'values') else np.asarray(y_train)

idx = np.arange(N)
try:
    pos = idx[yv == 1]
    neg = idx[yv == 0]
    k_pos = max(1, int(sample_size * (len(pos) / N)))
    k_neg = max(1, sample_size - k_pos)
    sel = np.concatenate([
        rng.choice(pos, size=min(k_pos, len(pos)), replace=False),
        rng.choice(neg, size=min(k_neg, len(neg)), replace=False)
    ])
    rng.shuffle(sel)
except Exception:
    sel = rng.choice(idx, size=sample_size, replace=False)

X_sample = X_train.iloc[sel].copy()
y_sample = y_train[sel].copy()   # ✅ FIX: y_train is a NumPy array, not a pandas Series

# ---- Try load cache if up-to-date (faster re-runs)
use_cache = False
if cache_path.exists() and cache_path.stat().st_mtime >= model_path.stat().st_mtime:
    try:
        cache = joblib.load(cache_path)
        if cache.get("shape") == X_sample.shape and cache.get("cols") == list(X_sample.columns):
            shap_values_arr = cache["shap_values"]
            base_values = cache["base_values"]
            feature_names = cache["cols"]
            X_used = X_sample
            use_cache = True
            print(f"[SHAP] Loaded cached SHAP arrays from {cache_path.name}")
    except Exception:
        pass

# ---- Compute SHAP (fast tree path, approximate) if no cache
if not use_cache:
    explainer = shap.TreeExplainer(
        model,
        feature_perturbation="tree_path_dependent",
        model_output="raw",
        approximate=True
    )
    sv = explainer(X_sample)
    shap_values_arr = sv.values
    base_values = sv.base_values
    feature_names = list(X_sample.columns)
    X_used = X_sample
    joblib.dump(
        {"shap_values": shap_values_arr, "base_values": base_values,
         "cols": feature_names, "shape": X_sample.shape},
        cache_path, compress=3
    )
    print(f"[SHAP] Computed & cached arrays → {cache_path.name}")

# ---- Wrap back into a SHAP Explanation object for plotting
sv_obj = shap.Explanation(
    values=shap_values_arr,
    base_values=base_values,
    data=X_used.values,
    feature_names=feature_names
)

# ---------- Plot 1: try 'num_leaves' (hyperparameter, likely absent) → robust fallback ----------
plt.figure()
title1 = "SHAP Scatter: num_leaves"
try:
    shap.plots.scatter(sv_obj[:, "num_leaves"], color=sv_obj, show=False)
except Exception:
    abs_means = np.abs(shap_values_arr).mean(axis=0)
    top_idx = int(abs_means.argmax())
    top_feat = feature_names[top_idx]
    title1 = f"SHAP Scatter (fallback): {top_feat}"
    shap.plots.scatter(sv_obj[:, top_feat], color=sv_obj, show=False)
plt.title(title1)
plt.savefig(out_dir / "shap_num_leaves_scatter.png", bbox_inches="tight")
plt.close()

# ---------- Plot 2: SHAP heatmap (limit display); fallback to bar if fails ----------
plt.figure()
try:
    shap.plots.heatmap(sv_obj, max_display=20, show=False)
    plt.title("SHAP Interaction Heatmap (top 20)")
except Exception:
    shap.plots.bar(sv_obj, max_display=20, show=False)
    plt.title("SHAP Summary (bar) — Fallback for Heatmap")
plt.savefig(out_dir / "shap_interaction_heatmap.png", bbox_inches="tight")
plt.close()

# ---------- Plot 3: Dependence ~ 'num_leaves' vs top partner (robust fallback) ----------
plt.figure()
title3 = "SHAP Dependence: num_leaves vs top interaction feature"
try:
    abs_means = np.abs(shap_values_arr).mean(axis=0)
    order = abs_means.argsort()
    partner_feat = feature_names[int(order[-1])]
    shap.plots.scatter(sv_obj[:, "num_leaves"], color=sv_obj[:, partner_feat], show=False)
except Exception:
    f1 = feature_names[int(order[-1])]
    f2 = feature_names[int(order[-2])] if len(order) > 1 else f1
    title3 = f"SHAP Dependence (fallback): {f1} vs {f2}"
    shap.plots.scatter(sv_obj[:, f1], color=sv_obj[:, f2], show=False)
plt.title(title3)
plt.savefig(out_dir / "shap_dependence_num_leaves.png", bbox_inches="tight")
plt.close()

print("[SHAP] Plots saved:",
      "staging/shap_num_leaves_scatter.png,",
      "staging/shap_interaction_heatmap.png,",
      "staging/shap_dependence_num_leaves.png")

# ---- Display the saved images inline (fast)
for fname, t in [
    ("shap_num_leaves_scatter.png", title1),
    ("shap_interaction_heatmap.png", "SHAP Interaction Heatmap (top 20)"),
    ("shap_dependence_num_leaves.png", title3),
]:
    fig = plt.figure(figsize=(7, 5))
    img = mpimg.imread(str(out_dir / fname))
    plt.imshow(img)
    plt.axis("off")
    plt.title(t)
    plt.show()


# --- Flexible Feature Weighting (minimal patch) ---
# You can assign weights to any subset of features; unspecified features default to 1.0.
FEATURE_WEIGHTS = {  # e.g., {"ttl": 0.1, "payload_byte_1": 0.5}
    # "ttl": 0.1,
}
def _apply_feature_weights(df_or_arr, feature_names=None, weights=FEATURE_WEIGHTS):
    """
    Minimal intrusive scaler: multiply specified columns by given weights.
    - If df_or_arr is a pandas DataFrame: use column names directly.
    - If it's a NumPy array: use feature_names to locate columns.
    """
    try:
        import pandas as _pd
        if hasattr(df_or_arr, "columns"):
            _w = {k: float(v) for k, v in (weights or {}).items()}
            for k, v in _w.items():
                if k in df_or_arr.columns:
                    df_or_arr[k] = df_or_arr[k] * v
            return df_or_arr
        else:
            # array path
            if feature_names is None or not weights:
                return df_or_arr
            name_to_idx = {n: i for i, n in enumerate(feature_names)}
            out = df_or_arr.copy()
            for k, v in weights.items():
                if k in name_to_idx:
                    out[:, name_to_idx[k]] *= float(v)
            return out
    except Exception:
        return df_or_arr


# --- Optional PCA step (minimal patch) ---
USE_PCA = True  # set False to disable without restructuring any pipeline
PCA_VARIANCE = 0.95

_pca_fitted = None
try:
    from sklearn.decomposition import PCA as _PCA
except Exception:
    _PCA = None

def _maybe_fit_pca(X_train):
    global _pca_fitted
    if not USE_PCA or _PCA is None:
        return None
    _pca_fitted = _PCA(n_components=PCA_VARIANCE, svd_solver="full", random_state=42)
    _pca_fitted.fit(X_train)
    return _pca_fitted

def _maybe_apply_pca(X_any):
    if not USE_PCA or _PCA is None or _pca_fitted is None:
        return X_any
    return _pca_fitted.transform(X_any)


## 9. Champion Auto-Wire → README & Model Card

Write the current **champion** (highest AUPRC) into `README.md` and `model_card.md`.  
Also saves `staging/champion_metrics.csv` for diffs.


In [ ]:
# ======================================================
# Cell 23 — Champion Auto-Wire → README & Model Card
# What:
#   • Select champion (max AUPRC) among staged models.
#   • Update README.md and model_card.md with current champion block.
#   • Write CSV snapshot for diffs/CI.
# Why:
#   • Keeps documentation in sync with latest results automatically.
# ======================================================
print(">>> Section 9: Champion Auto-Wire")
import re

candidates = [
    'bo_enhanced', 'bo_lgb', 'random_search', 'manual_grid',
    'ensemble_stack', 'ensemble_soft', 'ensemble_rf',
    'baseline', 'cnn1d'
]

def load_manifest(stage):
    """Load staged manifest if available; otherwise return None."""
    p = stage_dir(stage) / 'manifest.json'
    return json.loads(p.read_text()) if p.exists() else None

# Pick champion by AUPRC (tie-breaker: first encountered)
best = None
for s in candidates:
    m = load_manifest(s)
    if m and 'metrics' in m and 'AUPRC' in m['metrics']:
        ap = m['metrics']['AUPRC']
        if best is None or ap > best['metrics'].get('AUPRC', -1):
            best = m | {'stage': s}

if best is None:
    print("[AUTO-WIRE] No staged metrics found. Run training/evals first.")
else:
    ap = best['metrics']['AUPRC']
    f1 = best['metrics'].get('F1')
    tau = best['metrics'].get('tau')
    stamp = time.strftime('%Y-%m-%d %H:%M:%SZ', time.gmtime())

    # Markdown block to inject (safe indentation for Params)
    block = (
        f"### Champion: {best['model_name']} (stage: `{best['stage']}`)\n"
        f"- **Dataset**: {best.get('dataset','n/a')}\n"
        f"- **AUPRC**: {ap:.4f} | **F1@τ**: {f1:.4f} | **τ**: {tau:.3f}\n"
        f"- **Seed**: {best.get('seed','n/a')} | **Updated**: {stamp}\n"
        f"- **Params**:\n\n"
        f"```json\n{json.dumps(best.get('best_params', best.get('params', {})), indent=2)}\n```\n"
    )

    # --- Update README.md --------------------------------------------------
    rd = Path("README.md")
    rd_text = rd.read_text() if rd.exists() else "# Label Trainer\n"

    if "## Results" in rd_text:
        rd_text = re.sub(
            r"(## Results\s*\n)([\s\S]*?)(?=\n## |\Z)",
            rf"\1\n{block}\n",
            rd_text, flags=re.M
        )
    else:
        rd_text += "\n## Champion Metrics\n\n" + block + "\n"

    rd.write_text(rd_text)
    print("[AUTO-WIRE] README.md updated")

    # --- Update model_card.md (look for '## Performance') ------------------
    mc = Path("model_card.md")
    mc_text = mc.read_text() if mc.exists() else "# Model Card\n"

    if "## Performance" in mc_text:
        mc_text = re.sub(
            r"(## Performance\s*\n)([\s\S]*?)(?=\n## |\Z)",
            rf"\1\n{block}\n",
            mc_text, flags=re.M
        )
    else:
        mc_text += "\n## Performance\n\n" + block + "\n"

    mc.write_text(mc_text)
    print("[AUTO-WIRE] model_card.md updated")

    # --- Snapshot CSV ------------------------------------------------------
    out = pd.DataFrame([{'stage': best['stage'], **best['metrics']}])
    out.to_csv("staging/champion_metrics.csv", index=False)
    print("[AUTO-WIRE] staging/champion_metrics.csv written")


# --- Flexible Feature Weighting (minimal patch) ---
# You can assign weights to any subset of features; unspecified features default to 1.0.
FEATURE_WEIGHTS = {  # e.g., {"ttl": 0.1, "payload_byte_1": 0.5}
    # "ttl": 0.1,
}
def _apply_feature_weights(df_or_arr, feature_names=None, weights=FEATURE_WEIGHTS):
    """
    Minimal intrusive scaler: multiply specified columns by given weights.
    - If df_or_arr is a pandas DataFrame: use column names directly.
    - If it's a NumPy array: use feature_names to locate columns.
    """
    try:
        import pandas as _pd
        if hasattr(df_or_arr, "columns"):
            _w = {k: float(v) for k, v in (weights or {}).items()}
            for k, v in _w.items():
                if k in df_or_arr.columns:
                    df_or_arr[k] = df_or_arr[k] * v
            return df_or_arr
        else:
            # array path
            if feature_names is None or not weights:
                return df_or_arr
            name_to_idx = {n: i for i, n in enumerate(feature_names)}
            out = df_or_arr.copy()
            for k, v in weights.items():
                if k in name_to_idx:
                    out[:, name_to_idx[k]] *= float(v)
            return out
    except Exception:
        return df_or_arr


# --- Optional PCA step (minimal patch) ---
USE_PCA = True  # set False to disable without restructuring any pipeline
PCA_VARIANCE = 0.95

_pca_fitted = None
try:
    from sklearn.decomposition import PCA as _PCA
except Exception:
    _PCA = None

def _maybe_fit_pca(X_train):
    global _pca_fitted
    if not USE_PCA or _PCA is None:
        return None
    _pca_fitted = _PCA(n_components=PCA_VARIANCE, svd_solver="full", random_state=42)
    _pca_fitted.fit(X_train)
    return _pca_fitted

def _maybe_apply_pca(X_any):
    if not USE_PCA or _PCA is None or _pca_fitted is None:
        return X_any
    return _pca_fitted.transform(X_any)


>>> Section 9: Champion Auto-Wire
[AUTO-WIRE] README.md updated
[AUTO-WIRE] model_card.md updated
[AUTO-WIRE] staging/champion_metrics.csv written


# %% [markdown]
### Patch v3.2 — Minimal, academically-instrumented updates (auto-generated on 2025-08-29T23:01:26.578287Z)
- Flexible Feature Weighting (Section 3b): dictionary-driven column weights.
- Optional PCA in preprocessing (retain 95% variance), guard `USE_PCA=True`.
- LightGBM: early stopping (`early_stopping_rounds=50`) and widened BO search space; added `min_child_weight`.
- CNN baseline: micro grid over `filters` and `kernel_size` with a residual block.
- All changes inserted with minimal intrusion; original comments and structure retained.


### v3.4 Note — TTL preserved in Section 3b
- Removed any code that dropped `ttl`; feature is now retained and can be weighted via `FEATURE_WEIGHTS`.


### v3.4 Update — Image Saving
- All matplotlib plots are now saved under the `images/` folder in addition to being displayed.
